In [17]:
import torch
import torch.nn as nn
from transformers import BertModel
from transformers import AutoTokenizer
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import os
import duckdb as db
from dotenv import load_dotenv
from sklearn.model_selection import train_test_split

In [13]:
load_dotenv()

db_path = os.getenv("DATABASE_PATH")

conn = db.connect(db_path)

df = conn.sql("""select title, text, is_fake FROM processed_news
""").df()
conn.close()

In [16]:
df['full_text'] = df['title'] + " " + df['text']
X = df['full_text']
y = df['is_fake']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [21]:
df

,title,text,is_fake,full_text
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,1,Donald Trump Sends Out Embarrassing New Year’...
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,1,Drunk Bragging Trump Staffer Started Russian ...
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",1,Sheriff David Clarke Becomes An Internet Joke...
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",1,Trump Is So Obsessed He Even Has Obama’s Name...
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,1,Pope Francis Just Called Out Donald Trump Dur...
...,...,...,...,...
39094,'Fully committed' NATO backs new U.S. approach...,BRUSSELS (Reuters) - NATO allies on Tuesday we...,0,'Fully committed' NATO backs new U.S. approach...
39095,LexisNexis withdrew two products from Chinese ...,"LONDON (Reuters) - LexisNexis, a provider of l...",0,LexisNexis withdrew two products from Chinese ...
39096,Minsk cultural hub becomes haven from authorities,MINSK (Reuters) - In the shadow of disused Sov...,0,Minsk cultural hub becomes haven from authorit...
39097,Vatican upbeat on possibility of Pope Francis ...,MOSCOW (Reuters) - Vatican Secretary of State ...,0,Vatican upbeat on possibility of Pope Francis ...


In [51]:
class FakeNewsDataset(Dataset):
    def __init__(self, texts, labels, model_name = "bert-base-uncased", max_len = 256):
        super().__init__()
        self.texts = texts
        self.labels = labels
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts.iloc[idx])
        label = self.labels.iloc[idx]

        tokens = self.tokenizer(text, add_special_tokens=True, truncation = True, padding = "max_length", max_length=self.max_len, return_tensors='pt')

        item = {key: tensor.squeeze(0) for key, tensor in tokens.items()}

        item['label'] = torch.tensor(label, dtype=torch.long)

        return item

In [52]:
train_dataset = FakeNewsDataset(X_train, y_train)
test_dataset = FakeNewsDataset(X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)

In [53]:
class BertFakeNewsClassifier(nn.Module):
    def __init__(self, model_name="bert-base-uncased", drop_rate=0.3):
        super().__init__()
        self.model = BertModel(model_name)
        self.dropout = nn.Dropout(drop_rate)
        self.classifier = nn.Linear(self.bert.config.hidden_size, 2)

    def forward(self, input_ids, attenion_mask):
        outputs = self.model(input_ids, attenion_mask)
        cls_output = outputs.pooler_output
        x = self.dropout(cls_output)
        x = self.classifier(x)

        return x